# 38 - Full-cohort prefix-only versus reduced-strength P&P, worker 1/2

This is shard 1 of 2 over the exact corrected 1,300-identity LIBERO-PRO cohort: 13 suites x 10 tasks x init indices 0-9. Every identity runs two paired arms.

**Prefix-only** updates latent action positions 0-9 during each inner K-step P&P loop while positions 10-49 retain their ordinary sampler state. **Reduced-strength** updates the full 50-position latent but moves only `beta=0.5` toward each inner predict/perturb proposal. Both use K=5, Euler steps `(3,4)`, refine-last, and execute 10 actions.

Each worker runs 650 identities x 2 arms = 1,300 rollouts. Every 20 rollouts, the notebook prints balanced per-suite SR for both live arms beside exact historical full-PnP and unrefined 10-action SR, followed by U10, U20, and full-chunk uncertainty means.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable paired collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (SOURCE_PREFIX_STRENGTH_EXPERIMENT,
    load_bootstrap_manifest, run_source_prefix_strength_worker)

drive.mount('/content/drive')

EPISODES_PER_TASK = 10
INNER_STRENGTH = 0.5
SHARD_COUNT = 2
SHARD_INDEX = 1
EPISODE_LIMIT = None
EXPERIMENT = SOURCE_PREFIX_STRENGTH_EXPERIMENT
MANIFEST_PATH = Path(
    '/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json')
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest['source_model'] == PI05_REPO_ID, manifest['source_model']
SOURCE_MODEL_REVISION = manifest['source_model_revision']
assert SOURCE_MODEL_REVISION, 'v2 manifest is missing source_model_revision'

print({'experiment': EXPERIMENT, 'arms': ['prefix-only', 'inner-strength beta=0.5'],
       'episodes_per_task': EPISODES_PER_TASK, 'full_cohort_identities': 1300,
       'identities_in_this_shard': 650, 'rollouts_in_this_shard': 1300,
       'inner_strength': INNER_STRENGTH,
       'shard_count': SHARD_COUNT, 'shard_index': SHARD_INDEX,
       'episode_limit': EPISODE_LIMIT, 'manifest_hash': manifest['manifest_hash'],
       'source_model_revision': SOURCE_MODEL_REVISION})
run_source_prefix_strength_worker(
    episodes_per_task=EPISODES_PER_TASK, episode_limit=EPISODE_LIMIT,
    inner_strength=INNER_STRENGTH,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest['manifest_hash'],
    source_model_revision=SOURCE_MODEL_REVISION, experiment=EXPERIMENT)